# Notebook 0 — Hugging Face Authentication
**Adapted for Jupyter Notebook / Kubeflow**

This notebook demonstrates how to securely authenticate with the Hugging Face Hub
and load a model for inference. Three methods are shown, from most to least secure.

> **Jupyter / Kubeflow users:** Method 1 (Environment Variable) is the recommended approach.
> Colab Secrets are not available outside Google Colab.

## Authentication Methods

| Method | Security | Recommended for |
|---|---|---|
| **Method 1: Environment Variable** | ✅ Most secure | Jupyter, Kubeflow, any local setup |
| **Method 2: getpass (manual entry)** | ⚠️ Moderate | One-off runs, no token storage |
| **Method 3: Hardcoded in cell** | ❌ Insecure | Never — demonstration only |

> **Before running this notebook:** Set your HF token as an environment variable in your terminal:
>
> ```bash
> export HF_TOKEN=your_token_here          # Mac / Linux / Kubeflow Terminal
> set HF_TOKEN=your_token_here             # Windows Command Prompt
> $env:HF_TOKEN = "your_token_here"        # Windows PowerShell
> ```
>
> On Kubeflow, add it to `~/.bashrc` to persist across sessions:
> ```bash
> echo 'export HF_TOKEN=your_token_here' >> ~/.bashrc && source ~/.bashrc
> ```

## Method 1: Environment Variable (RECOMMENDED for Jupyter / Kubeflow)

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers", "huggingface_hub", "accelerate",
                "datasets", "sentencepiece", "tiktoken",
                "langdetect", "pyarrow", "ipywidgets", "-q"])
print("✅ All packages installed")

✅ All packages installed


In [2]:
import os
import subprocess
result = subprocess.run(
    ["bash", "-c", "source ~/.bashrc && echo $HF_TOKEN"],
    capture_output=True, text=True
)
os.environ["HF_TOKEN"] = result.stdout.strip()

In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Load HF token from environment variable — set this in your terminal before launching Jupyter
# Mac/Linux/Kubeflow:  export HF_TOKEN=your_token_here
# Windows CMD:         set HF_TOKEN=your_token_here
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN environment variable is not set.\n"
        "Run this in your terminal before launching Jupyter:\n"
        "  export HF_TOKEN=your_token_here"
    )

# Authenticate with HuggingFace Hub
login(token=HF_TOKEN)
print("✅ Authenticated with Hugging Face Hub")

# Load GPT-2 model and tokenizer
model_name = "gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model      = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN,
    torch_dtype=torch.float32,   # float32 for CPU — do NOT use float16 on CPU
    device_map="cpu"
)
print(f"✅ Model loaded: {model_name}")
print(f"   Device: {next(model.parameters()).device}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Authenticated with Hugging Face Hub


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded: gpt2
   Device: cpu


### Text Generation — verify the model works

In [4]:
input_text = "Machine learning is revolutionizing"
inputs     = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=50,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nPrompt   : {input_text}")
print(f"Generated: {generated_text}")


Prompt   : Machine learning is revolutionizing
Generated: Machine learning is revolutionizing the way we think about technology and how we interact with it.

The next great thing for you is the ability to build a machine learning model that can make decisions. You can build a machine learning model that can be


## Method 2: getpass — Manual Entry (Semi-Secure)

In [ ]:
from getpass import getpass

# Token is typed interactively — not stored in the notebook
# Use this if you cannot set an environment variable
HF_TOKEN_INPUT = getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN_INPUT)
print("✅ Authenticated via manual input")

## Method 3: Hardcoded Token (⚠️ INSECURE — Demonstration Only)

> **Warning:** Never use this in a shared notebook or commit it to a repository.
> If your token is exposed, revoke it immediately at https://huggingface.co/settings/tokens

In [ ]:
# ⚠️ INSECURE — FOR DEMONSTRATION ONLY
# Replace the placeholder below with your token ONLY for isolated local testing
# Delete or comment out before sharing any notebook

HF_TOKEN_DIRECT = "hf_replace_with_your_token"  # ← NEVER share this

# login(token=HF_TOKEN_DIRECT)
# Commented out to prevent accidental execution
print("⚠️  Method 3 is shown for awareness only. Use Method 1 in practice.")

## Summary

| Concept | Detail |
|---|---|
| `os.environ.get("HF_TOKEN")` | Reads token from shell environment — the safe Jupyter/Kubeflow approach |
| `login(token=...)` | Authenticates the session with HuggingFace Hub |
| `torch.float32` | Use on CPU — float16 is for GPU only and will cause errors on CPU |
| `device_map="cpu"` | Forces model to CPU — appropriate for Kubeflow CPU-only nodes |
| `getpass` | Interactive, hidden input — token never stored in notebook output |